# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for every exercise in [1.5-pandas-exercises.ipynb](1.5-pandas-exercises.ipynb), including the eleven-question earthquake walkthrough.
:::

## Exercise 1: A DataFrame and a column mean

Build a DataFrame with columns `temp_celsius = [18.2, 17.5, 19.1, 16.8]` and `discharge_m3s = [48.0, 51.2, 47.5, 53.1]`. Print the column dtypes and the mean temperature, rounded to two decimals.

In [ ]:
import pandas as pd
df = pd.DataFrame({"temp_celsius": [18.2, 17.5, 19.1, 16.8],
                   "discharge_m3s": [48.0, 51.2, 47.5, 53.1]})
print(df.dtypes.to_dict())
print(round(df["temp_celsius"].mean(), 2))

## Exercise 2: Write and read a CSV with dates

Build a DataFrame with a `date` column (`"2024-06-01"`, `"2024-06-02"`, `"2024-06-03"`) and a `temp_celsius` column. Write it to `_files/obs.csv` without the index, then read it back with `parse_dates=["date"]` and print the dtypes.

In [ ]:
import pandas as pd
from pathlib import Path

Path("_files").mkdir(exist_ok=True)
frame = pd.DataFrame({"date": ["2024-06-01", "2024-06-02", "2024-06-03"],
                      "temp_celsius": [18.2, 17.5, 19.1]})
frame.to_csv("_files/obs.csv", index=False)
obs = pd.read_csv("_files/obs.csv", parse_dates=["date"])
print(obs.dtypes.to_dict())
print(obs.head())

## Exercise 3: Label, position, and mask

Given

```python
df = pd.DataFrame({"station": ["BAS", "LUG", "JFJ"], "temp_celsius": [18.0, 21.0, -1.0]},
                  index=["a", "b", "c"])
```

print the temperature at label `"b"`, the whole first row by position, and all rows whose temperature is below 0 °C.

In [ ]:
import pandas as pd
df = pd.DataFrame({"station": ["BAS", "LUG", "JFJ"], "temp_celsius": [18.0, 21.0, -1.0]},
                  index=["a", "b", "c"])
print(df.loc["b", "temp_celsius"])     # by label
print(df.iloc[0])                      # by position
print(df[df["temp_celsius"] < 0.0])    # boolean mask

## Exercise 4: `.isel` and `.sel` on a DataArray

The block below stores ten days of temperature at three stations twice: as an xarray `DataArray` with dimensions `time` and `station`, and as a DataFrame with the dates as rows and the stations as columns. Using `temp_da`, print:

1. every station's first day, by position with `.isel` — and the same row of `temp_df` with `.iloc`;
2. JFJ on 5 June, by label with `.sel` — and the same cell of `temp_df` with `.loc`;
3. LUG's last three days — `.isel` also accepts a list of positions;
4. the BAS reading nearest to 18:00 on 7 June, and the date it was taken. Select the station first and apply `method="nearest"` to the time alone: `.sel` applies the method to every dimension it is given, and the station names cannot be "nearest" to anything.

In a comment, say why item 4 returns the day it does, and what `temp_da.isel(station=0)` would return if the array had been built with its dimensions in the other order, `("station", "time")`.

```python
import xarray as xr

rng = np.random.default_rng(2)
dates = pd.date_range("2024-06-01", periods=10, freq="D")
temp_celsius = np.array([18.0, 21.0, -1.0]) + rng.normal(0, 1.5, size=(10, 3))   # (time, station)

temp_da = xr.DataArray(temp_celsius, dims=("time", "station"),
                       coords={"time": dates, "station": ["BAS", "LUG", "JFJ"]},
                       attrs={"units": "degC"})
temp_df = pd.DataFrame(temp_celsius, index=dates, columns=["BAS", "LUG", "JFJ"])
```

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

rng = np.random.default_rng(2)
dates = pd.date_range("2024-06-01", periods=10, freq="D")
temp_celsius = np.array([18.0, 21.0, -1.0]) + rng.normal(0, 1.5, size=(10, 3))   # (time, station)

temp_da = xr.DataArray(temp_celsius, dims=("time", "station"),
                       coords={"time": dates, "station": ["BAS", "LUG", "JFJ"]},
                       attrs={"units": "degC"})
temp_df = pd.DataFrame(temp_celsius, index=dates, columns=["BAS", "LUG", "JFJ"])

# 1. first day, by position
print(temp_da.isel(time=0).values)
print(temp_df.iloc[0].values)

# 2. JFJ on 5 June, by label
print(temp_da.sel(station="JFJ", time="2024-06-05").item())
print(temp_df.loc["2024-06-05", "JFJ"])

# 3. LUG's last three days, by a list of positions
print(temp_da.sel(station="LUG").isel(time=[7, 8, 9]).values)

# 4. nearest to 18:00 on 7 June: the station first, then the time alone
nearest = temp_da.sel(station="BAS").sel(time="2024-06-07 18:00", method="nearest")
print(nearest["time"].values, nearest.item())

# 18:00 on 7 June is six hours before 8 June and eighteen after 7 June, so method="nearest"
# returns 8 June. Built as ("station", "time"), temp_da.isel(station=0) would still return BAS's
# ten days: xarray finds a dimension by its name, not by its place in the array

## Exercise 5: Resample and rolling

Build a 40-day daily Series indexed by date (`np.random.default_rng(0)`, mean 18, std 2). Print the monthly means and the first five values of the 3-day rolling mean, both rounded to two decimals.

In [ ]:
import numpy as np
import pandas as pd
dates = pd.date_range("2024-06-01", periods=40, freq="D")
s = pd.Series(18 + np.random.default_rng(0).normal(0, 2, 40), index=dates)
print(s.resample("MS").mean().round(2).tolist())
print(s.rolling(window=3).mean().round(2).head(5).tolist())

## Exercise 6: Group and aggregate

Given

```python
df = pd.DataFrame({"station": ["BAS", "BAS", "LUG", "LUG"],
                   "temp_celsius": [18.0, 19.0, 21.0, 22.0]})
```

compute the mean temperature per station and print it as a dict.

In [ ]:
import pandas as pd
df = pd.DataFrame({"station": ["BAS", "BAS", "LUG", "LUG"],
                   "temp_celsius": [18.0, 19.0, 21.0, 22.0]})
print(df.groupby("station")["temp_celsius"].mean().to_dict())

## Exercise 7: Handle a gap three ways

Given `s = pd.Series([1.0, np.nan, np.nan, 4.0, 5.0])`, print the forward-filled series, the linearly interpolated series, and the NaN-skipping mean. Note in a comment why the three differ.

In [ ]:
import numpy as np
import pandas as pd
s = pd.Series([1.0, np.nan, np.nan, 4.0, 5.0])
print(s.ffill().tolist())          # carries the last known value forward
print(s.interpolate().tolist())    # straight line between known neighbours
print(round(s.mean(), 2))          # mean of the present values only
# ffill repeats 1.0; interpolate ramps 1->4; mean ignores the gaps entirely

## Exercise 8: Join two tables

Given an observations table and a metadata table that share a `station` key, left-join the elevation onto the observations and print the result.

```python
obs = pd.DataFrame({"station": ["BAS", "LUG"], "temp_celsius": [18.0, 21.0]})
meta = pd.DataFrame({"station": ["BAS", "LUG"], "elevation_m": [316, 273]})
```

Then stack the row below — a third station that reports only temperature — onto the joined table with `concat`, and print the result. Which cell comes out missing, and why?

```python
jfj = pd.DataFrame({"station": ["JFJ"], "temp_celsius": [-1.0]})
```

In [ ]:
import pandas as pd
obs = pd.DataFrame({"station": ["BAS", "LUG"], "temp_celsius": [18.0, 21.0]})
meta = pd.DataFrame({"station": ["BAS", "LUG"], "elevation_m": [316, 273]})
joined = obs.merge(meta, on="station", how="left")
print(joined)

jfj = pd.DataFrame({"station": ["JFJ"], "temp_celsius": [-1.0]})
stacked = pd.concat([joined, jfj], ignore_index=True)
print(stacked)
# JFJ's elevation_m is NaN: the new row has no elevation column, and concat marks the missing
# cell as NaN rather than inventing a value

## Exercise 9: A log-scale axis, and counting past a threshold

River discharge is a classic right-skewed variable: most days sit near base flow, with a long tail of rare, much larger flood events.

```python
rng = np.random.default_rng(1)
discharge_m3s = pd.Series(np.exp(rng.normal(3.4, 1.3, 300)))
```

1. Plot a histogram of `discharge_m3s` with `.plot(kind="hist")`, on default (linear) axes, then again with `logy=True`. Which view makes the long tail easier to read?
2. Build a boolean mask for discharge above 100 m³ s⁻¹ (a "flood" threshold), filter `discharge_m3s` with it, and print how many of the 300 values exceed it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
discharge_m3s = pd.Series(np.exp(rng.normal(3.4, 1.3, 300)))

fig, ax = plt.subplots(figsize=(5, 3))
discharge_m3s.plot(ax=ax, kind="hist", color="tab:blue")
ax.set_xlabel("discharge (m$^3$ s$^{-1}$)")
ax.set_title("linear count axis")
plt.show()

fig, ax = plt.subplots(figsize=(5, 3))
discharge_m3s.plot(ax=ax, kind="hist", color="tab:blue", logy=True)
ax.set_xlabel("discharge (m$^3$ s$^{-1}$)")
ax.set_title("log-scale count axis")
plt.show()

# the linear view crushes every bin above the peak into almost nothing; logy=True keeps
# the rare, large-discharge bins visible instead of flattening them against the x-axis

flood_mask = discharge_m3s > 100.0
floods = discharge_m3s[flood_mask]
print("days above 100 m³ s⁻¹:", len(floods))

## Exercise 10: Filtered vs. unfiltered histogram

```python
rng = np.random.default_rng(0)
temp_celsius = pd.Series(rng.normal(18.0, 2.0, 200))
temp_celsius[:5] = -999.0     # a stuck sensor: five obviously invalid readings
```

1. Plot a histogram of the raw `temp_celsius` values with `.plot(kind="hist")`. What does the stuck sensor do to the plot?
2. Build a boolean mask that keeps only values above −50 °C, filter the Series with it, and plot the histogram again. In one comment, say which of the two plots you would trust.
3. Finally, mark the stuck readings in `temp_celsius` itself as missing: set every value below −50 °C to `np.nan` with a masked `.loc` assignment, then print the mean of the result next to the mean of the filtered Series from step 2. Why do the two agree?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
temp_celsius = pd.Series(rng.normal(18.0, 2.0, 200))
temp_celsius[:5] = -999.0

fig, ax = plt.subplots(figsize=(5, 3))
temp_celsius.plot(ax=ax, kind="hist", color="tab:red")
ax.set_xlabel("temperature (°C)")
ax.set_title("raw (unfiltered)")
plt.show()

mask = temp_celsius > -50.0
filtered = temp_celsius[mask]

fig, ax = plt.subplots(figsize=(5, 3))
filtered.plot(ax=ax, kind="hist", color="tab:red")
ax.set_xlabel("temperature (°C)")
ax.set_title("filtered")
plt.show()

# the five stuck readings pull the x-axis out to -999, squashing the real distribution
# into one thin bin; the filtered plot is the one worth trusting

temp_celsius.loc[temp_celsius < -50.0] = np.nan     # mark the stuck readings as missing, in place
print(round(temp_celsius.mean(), 3), round(filtered.mean(), 3))
# the two agree: mean skips NaN, so marking the readings missing and dropping them leave the
# same 195 values in the average

## Exercise 11: Earthquake data analysis

```{figure} _static/usgs-2014-south-napa-earthquake.jpg
---
name: south-napa-surface-rupture
width: 800px
alt: A dirt track through a vineyard, broken along its length by a continuous ridge of pushed-up soil following the fault trace
---
Surface rupture from the magnitude 6.0 South Napa earthquake of 24 August 2014 — the kind of event a row in this catalog stands for. The continuous "mole track" running parallel to the strike of the fault indicates some east–west compression on top of the <a href="https://www.usgs.gov/faqs/what-fault-and-what-are-different-types">right-lateral faulting</a>; taken near Buhman Road. Photo <a href="https://www.usgs.gov/media/images/2014-south-napa-ca-m6-earthquake-august-24-6">USGS</a>, public domain.
```

This exercise reviews the `pandas` fundamentals of this subchapter on one real catalog, covering how to

* open csv files
* manipulate dataframe indexes
* parse date columns
* examine basic dataframe statistics
* manipulate text columns and extract values
* plot dataframe contents using
  * bar charts
  * histograms
  * scatter plots

The data is a snapshot of the [USGS Earthquakes Database](https://earthquake.usgs.gov/earthquakes/search/): every event recorded worldwide in 2014, 120 108 rows.

You do not need to download the file yourself. The pre-supplied cell below fetches and caches it with the help of [pooch](https://www.fatiando.org/pooch/latest/) — there is no need to read pooch's documentation unless you want to — and stores the path to the file in the variable `datafile`.


In [ ]:
# Pre-supplied: download and cache the earthquake data.
import pooch

datafile = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2014.csv",
    known_hash="sha256:84d455fb96dc8f782fba4b5fbe56cb8970cab678f07c766fcba1b1c4674de1b1",
    fname="usgs_earthquakes_2014.csv",
    path=pooch.os_cache("mlees"),
)

**Q1)** Imports, and a display option so wide tables stay readable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

**Q2)** Read it as it comes, and look at what you got.

In [ ]:
df = pd.read_csv(datafile)
print(df.shape)
print(df.head())
df.info()

# the time and updated columns came in as plain strings, not dates: pandas does not guess

**Q3)** Re-read with the dates parsed and the event id as the index.

In [ ]:
df = pd.read_csv(datafile, parse_dates=["time", "updated"], index_col="id")
print(df.head())
df.info()

**Q4)** Basic statistics of every numeric column.

In [ ]:
print(df.describe())

**Q5)** The twenty largest events by magnitude.

In [ ]:
print(df.nlargest(20, "mag")[["place", "mag", "depth"]])

**Q6)** Split `place` on the comma and keep the second piece as `country`.

In [ ]:
country = df["place"].str.split(",", expand=True, n=1)
df["country"] = country[1]
print(df[["place", "country"]].head())

**Q7)** The distinct values of the new column.

In [ ]:
print(df["country"].unique()[:12])
print("distinct values:", df["country"].nunique())

# a plain split leaves the space that followed the comma, hence ' Alaska' rather than 'Alaska'

**Q8)** Everything above magnitude 4.

In [ ]:
df_filt = df[df["mag"] > 4]
print(df_filt.shape)

**Q9)** Count them, count them per location, and chart the top five.

In [ ]:
num1 = df_filt["mag"].count()
print("events above magnitude 4:", num1)

num2 = df_filt["country"].value_counts()
print(num2.head())

top5_df = pd.DataFrame({"country": list(num2.index[:5]),
                        "earthquake_num": list(num2.values[:5])})
print(top5_df)

fig, ax = plt.subplots(figsize=(7, 3.2))
# x and y name the columns to plot; rot=0 keeps the labels horizontal
top5_df.plot(kind="bar", x="country", y="earthquake_num", rot=0, ax=ax)
ax.set_ylabel("number of earthquakes")
plt.tight_layout()
plt.show()

**Q10)** The magnitude distribution, three ways.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
df["mag"].plot(kind="hist", bins=20, ax=ax)
ax.set_xlabel("magnitude")
ax.set_ylabel("number")
ax.grid(alpha=0.2, color="b", linestyle="--")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.2))
df["mag"].plot(kind="hist", bins=20, ax=ax)
ax.set_xlabel("magnitude")
ax.set_ylabel("number")
ax.grid(alpha=0.2, color="b", linestyle="--")
ax.set_yscale("log")          # or plot(kind="hist", logy=True), as in the lecture
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
df_filt["mag"].plot(kind="hist", bins=20, color="#1f77b4", ax=axes[0])
axes[0].set_title("filtered")
df["mag"].plot(kind="hist", bins=20, color="#ff7f0e", ax=axes[1])
axes[1].set_title("unfiltered")
for ax in axes:
    ax.set_xlabel("magnitude")
    ax.set_ylabel("number")
    ax.grid(alpha=0.2, color="b", linestyle="--")
plt.tight_layout()
plt.show()

**Q11)** Where the events are, coloured by magnitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, events, title in zip(axes, [df_filt, df], ["filtered", "unfiltered"]):
    sc = ax.scatter(events["longitude"], events["latitude"], c=events["mag"], s=4,
                    cmap="Reds", vmin=0, vmax=8)      # one colour range for both panels
    fig.colorbar(sc, ax=ax, label="magnitude")
    ax.set_xlabel("longitude (°E)")
    ax.set_ylabel("latitude (°N)")
    ax.set_title(title)
plt.tight_layout()
plt.show()

# Yes. Both panels trace the plate boundaries, but the unfiltered one also shows where the
# instruments are: dense pale clusters over California, Alaska, Oklahoma and Japan are small
# events that only a close network can record. The filtered map is closer to a map of
# tectonics; the unfiltered one is partly a map of seismometer coverage.